# ARA 10ppm Barges Window Report

Rebuilds the daily Platts eWindow report from Snowflake, including:

1. **Curves** — daily Platts MOPL differential (`AMOPM00`) with latest-print callout
2. **Barges** — daily volume + cycle-to-date Sankey flows
3. **Strip sections** — intraday bid/offer chart + counterparty table (FE / MW / BE)

Validated against the archived HTML report dated **2026-06-25**.

In [8]:
from datetime import date
from pathlib import Path

from report_config import ARA_10PPM_BARGES
from snowflake_data import fetch_window_orders
from chart_data import fetch_mopl_curve, fetch_daily_volume, fetch_cycle_flows, strip_title
from aggregate import build_counterparty_table, format_activity_table
from report_charts import build_all_charts, charts_to_html_images
from render_html import render_report

from datetime import datetime
import pandas as pd

REPORT_DATE = datetime.today().date() + pd.Timedelta(days=-1)#date(2026,6,18)#
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [9]:
REPORT_DATE

datetime.date(2026, 9, 21)

In [10]:
# --- chart inputs ---
mopl_df = fetch_mopl_curve(REPORT_DATE)
volume_df = fetch_daily_volume(ARA_10PPM_BARGES, REPORT_DATE)
flows_df = fetch_cycle_flows(ARA_10PPM_BARGES, REPORT_DATE)
day_orders = fetch_window_orders(ARA_10PPM_BARGES, REPORT_DATE)

strip_titles = {
    code: strip_title(day_orders, code)
    for code in ARA_10PPM_BARGES.strip_labels
}
print("Strip titles:", strip_titles)

Strip titles: {'FE': 'FE', 'MW': 'MW', 'BE': 'BE'}


In [11]:
charts = build_all_charts(
    ARA_10PPM_BARGES,
    REPORT_DATE,
    mopl_df,
    volume_df,
    flows_df,
    day_orders,
    strip_titles,
)
chart_images = charts_to_html_images(charts)

# preview in notebook
charts["curve"].show()
charts["daily_volume"].show()
charts["sankey"].show()
charts["intraday_FE"].show()

In [5]:
sections = {}
strip_codes = {}

for strip_code, strip_label in ARA_10PPM_BARGES.strip_labels.items():
    raw = fetch_window_orders(ARA_10PPM_BARGES, REPORT_DATE, strip=strip_code)
    sections[strip_label] = build_counterparty_table(raw)
    strip_codes[strip_label] = strip_code

    print(f"\n=== {strip_label} ===")
    display(format_activity_table(sections[strip_label]))


=== Front End ===


,MARKET_MAKER_MNEMONIC,Bids,Gets hit,Lifts,Bought,Offers,Gets lifted,Hits,Sold,VWAP
0,AGTEUSA,0,0,0,0,1,1,0,"3,000",0.75
1,BP,0,0,0,0,1,0,0,0,0.00
2,EXTAP,2,1,0,"3,000",0,0,0,0,0.00
3,GLENCOREUK,1,0,0,0,1,0,0,0,0.00
4,HARTREEUK,1,0,1,"3,000",0,0,0,0,0.75
5,MBENERGY,1,0,0,0,0,0,0,0,0.00
6,MERCURIASA,0,0,0,0,1,0,1,"3,000",0.00
7,STR,0,0,0,0,1,0,0,0,0.00
8,TOTSA,0,0,0,0,1,0,0,0,0.00
9,VITOL,1,0,0,0,0,0,0,0,0.00



=== Mid Window ===


,MARKET_MAKER_MNEMONIC,Bids,Gets hit,Lifts,Bought,Offers,Gets lifted,Hits,Sold,VWAP
0,AGTEUSA,0,0,0,0,1,0,0,0,0.00
1,BP,0,0,0,0,1,0,0,0,0.00
2,EXTAP,1,0,0,0,0,0,0,0,0.00
3,GLENCOREUK,1,0,0,0,1,0,0,0,0.00
4,MERCURIASA,0,0,0,0,1,0,0,0,0.00
5,STR,0,0,0,0,1,0,0,0,0.00
6,VITOL,1,0,0,0,0,0,0,0,0.00
7,TOTAL ROW,3,0,0,0,5,0,0,0,nan



=== Back End ===


,MARKET_MAKER_MNEMONIC,Bids,Gets hit,Lifts,Bought,Offers,Gets lifted,Hits,Sold,VWAP
0,BP,0,0,0,0,1,0,0,0,0.00
1,EXTAP,1,0,0,0,0,0,0,0,0.00
2,GLENCOREUK,1,0,0,0,1,0,0,0,0.00
3,MERCURIASA,0,0,0,0,1,0,0,0,0.00
4,STR,0,0,0,0,1,0,0,0,0.00
5,TOTSA,0,0,0,0,1,0,0,0,0.00
6,VITOL,1,0,0,0,0,0,0,0,0.00
7,TOTAL ROW,3,0,0,0,5,0,0,0,nan


In [6]:
html = render_report(
    ARA_10PPM_BARGES,
    REPORT_DATE.strftime("%d-%b-%y"),
    sections,
    chart_images,
    strip_codes,
)

out_path = OUTPUT_DIR / f"ara_10ppm_barges_{REPORT_DATE.isoformat()}.html"
out_path.write_text(html, encoding="utf-8")
print(f"Saved {out_path.resolve()}")

Saved C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\window_trades_report\output\ara_10ppm_barges_2026-06-18.html
